In [61]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import numpy as np
from torch import device, cuda, tensor, float32
from torch.utils.data import Dataset,DataLoader
from torchvision.models import densenet121,resnet50,efficientnet_v2_m,vit_b_16,swin_v2_t,DenseNet121_Weights,ResNet50_Weights,EfficientNet_V2_M_Weights,ViT_B_16_Weights,Swin_V2_T_Weights
import torchvision.transforms as transforms
from PIL import Image
from sklearn.model_selection import train_test_split
from warnings import filterwarnings
filterwarnings("ignore")

%matplotlib inline

dev = device("cuda" if cuda.is_available() else "cpu")
dev

device(type='cuda')

In [62]:
df = pd.read_csv(r"D:\Data\Data_Entry_2017.csv")
train_val_names = pd.read_csv(r"D:\Data\train_val_list.txt",header=None,names=["Image Index"])
test_names = pd.read_csv(r"D:\Data\test_list.txt",header=None,names=["Image Index"])

In [63]:
train_val_names

,Image Index
0,00000001_000.png
1,00000001_001.png
2,00000001_002.png
3,00000002_000.png
4,00000004_000.png
...,...
86519,00030789_000.png
86520,00030793_000.png
86521,00030795_000.png
86522,00030801_000.png


In [64]:
test_names

,Image Index
0,00000003_000.png
1,00000003_001.png
2,00000003_002.png
3,00000003_003.png
4,00000003_004.png
...,...
25591,00030800_000.png
25592,00030802_000.png
25593,00030803_000.png
25594,00030804_000.png


In [65]:
train_val_names = set(train_val_names['Image Index'].values)
test_names = set(test_names['Image Index'].values)

In [66]:
PATH = r"D:\Data"
IMG_FOLDERS = [os.path.join(PATH,x+"\\images") for x in os.listdir(PATH) if x.startswith("images_")] 

NUM_OF_CLASSES = 14

ALL_DISEASES = sorted(df[df["Finding Labels"] != "No Finding"]["Finding Labels"].str.split("|").explode().unique())
ALL_DISEASES

['Atelectasis',
 'Cardiomegaly',
 'Consolidation',
 'Edema',
 'Effusion',
 'Emphysema',
 'Fibrosis',
 'Hernia',
 'Infiltration',
 'Mass',
 'Nodule',
 'Pleural_Thickening',
 'Pneumonia',
 'Pneumothorax']

In [67]:
def get_transforms(size=224 , In_Train=True):
    if In_Train:
        return transforms.Compose([transforms.Resize((size,size)),
                                   transforms.RandomRotation(10),
                                   transforms.RandomHorizontalFlip(),
                                   transforms.ToTensor(),
                                   transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                                        std=[0.229, 0.224, 0.225])])
    else:
        return transforms.Compose([transforms.Resize((size,size)),
                                   transforms.ToTensor(),
                                   transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                                        std=[0.229, 0.224, 0.225])])

In [68]:
df_imgs_labels = df[["Image Index","Finding Labels"]]
df_imgs_labels

,Image Index,Finding Labels
0,00000001_000.png,Cardiomegaly
1,00000001_001.png,Cardiomegaly|Emphysema
2,00000001_002.png,Cardiomegaly|Effusion
3,00000002_000.png,No Finding
4,00000003_000.png,Hernia
...,...,...
112115,00030801_001.png,Mass|Pneumonia
112116,00030802_000.png,No Finding
112117,00030803_000.png,No Finding
112118,00030804_000.png,No Finding


In [69]:
train_val_df = df_imgs_labels[df_imgs_labels['Image Index'].isin(train_val_names)]
train_val_df

,Image Index,Finding Labels
0,00000001_000.png,Cardiomegaly
1,00000001_001.png,Cardiomegaly|Emphysema
2,00000001_002.png,Cardiomegaly|Effusion
3,00000002_000.png,No Finding
12,00000004_000.png,Mass|Nodule
...,...,...
112100,00030789_000.png,Infiltration
112106,00030793_000.png,Mass|Nodule
112108,00030795_000.png,Pleural_Thickening
112114,00030801_000.png,No Finding


In [70]:
train_df,val_df = train_test_split(train_val_df,test_size=0.2,random_state=44)
test_df = df_imgs_labels[df_imgs_labels['Image Index'].isin(test_names)]

print("Train Shape : ", train_df.shape)
print("Validation Shape : ", val_df.shape)
print("Test Shape : ", test_df.shape)

train_df.shape[0] + val_df.shape[0] + test_df.shape[0]

Train Shape :  (69219, 2)
Validation Shape :  (17305, 2)
Test Shape :  (25596, 2)


112120

In [71]:
class ChestXRayDataset(Dataset):
    def __init__(self,df,img_folders,transforms=None):
        self.df = df.reset_index(drop=True)
        self.img_folders = img_folders
        self.transforms = transforms
        self.labels = self._encode_labels()

    def _encode_labels(self):
        encode = []

        for label in self.df["Finding Labels"]:
            diseases_in_img = label.split("|")

            vector = [0.0] * NUM_OF_CLASSES
            for disease in diseases_in_img:
                if disease in ALL_DISEASES:
                    idx = ALL_DISEASES.index(disease)
                    vector[idx] = 1.0
            encode.append(vector)
        return encode
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, index):
        img_name = self.df.loc[index,"Image Index"]

        img = None
        for folder in IMG_FOLDERS:
            img_path = os.path.join(folder,img_name)
            if os.path.exists(img_path):
                img = Image.open(img_path).convert("RGB")
                break
        
        if self.transforms:
            img = self.transforms(img)

        label = tensor(self.labels[index],dtype=float32)
        return img,label


train_dataset = ChestXRayDataset(df_imgs_labels,IMG_FOLDERS,transforms=get_transforms())
val_dataset = ChestXRayDataset(df_imgs_labels,IMG_FOLDERS,get_transforms(In_Train=False))
test_dataset = ChestXRayDataset(df_imgs_labels,IMG_FOLDERS,get_transforms(In_Train=False))

eff_train_dataset = ChestXRayDataset(df_imgs_labels,IMG_FOLDERS,get_transforms(384))
eff_val_dataset = ChestXRayDataset(df_imgs_labels,IMG_FOLDERS,get_transforms(384,In_Train=False))
eff_test_dataset = ChestXRayDataset(df_imgs_labels,IMG_FOLDERS,get_transforms(384,In_Train=False))

swin_train_dataset = ChestXRayDataset(df_imgs_labels,IMG_FOLDERS,get_transforms(256))
swin_val_dataset = ChestXRayDataset(df_imgs_labels,IMG_FOLDERS,get_transforms(256,In_Train=False))
swin_test_dataset = ChestXRayDataset(df_imgs_labels,IMG_FOLDERS,get_transforms(256,In_Train=False))